In [1]:
import pandas as pd

In [32]:
orders = pd.read_csv("orders.csv")
orders_df = orders.copy()

In [33]:
orderlines = pd.read_csv('orderlines.csv')
orderlines_df = orderlines.copy()

In [34]:
products = pd.read_csv("products.csv")
products_df = products.copy()

In [35]:
brands = pd.read_csv("brands.csv")
brands_df = brands.copy()

* Given the time available and the impact on the final assessment, deciding on the following cleaning methods

# Orders Table

226909 rows × 4 columns
No Duplicates
total_paid - 5 missing values - to drop?
created_date convert to datetime series

In [8]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 6.9 MB


In [7]:
orders_df.duplicated().sum()
# No duplicates

np.int64(0)

In [10]:
orders_df.isnull().sum()
# total_paid has 5 missing values

order_id        0
created_date    0
total_paid      5
state           0
dtype: int64

In [36]:
orders_df["total_paid"].isnull().mean()*100
# Only .22% of data is missing, hence deleting these rows

np.float64(0.0022035265238487677)

In [37]:
orders_df.dropna(subset=["total_paid"], inplace = True)

In [39]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

In [20]:
orders_df.info()

<class 'pandas.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[us]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 8.7 MB


# Orderlines Table

No duplicates
product_id - value is 0, so drop column
product_quantity - 
• 10 rows with quantity > 100 - flagged to review
• Example: 192 units of APP1662 at €519 each = €99,648 in a single orderline
date - convert to datetime series
sku - brand code has lower case par0072(12 times). Convert sku column to uppercase
unit_price - 12% double decimal data. to drop or Regex?
unit_price to numeric conversion

In [21]:
orderlines_df.duplicated().sum()
# No duplicates

np.int64(0)

In [ ]:
orderlines_df.isnull().sum()
# No null values

In [40]:
orderlines_df.info()
# unit_price str to num
# date str to datetime series

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                293983 non-null  int64
 1   id_order          293983 non-null  int64
 2   product_id        293983 non-null  int64
 3   product_quantity  293983 non-null  int64
 4   sku               293983 non-null  str  
 5   unit_price        293983 non-null  str  
 6   date              293983 non-null  str  
dtypes: int64(4), str(3)
memory usage: 15.7 MB


In [41]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

In [42]:
orderlines_df["date"].dtype

dtype('<M8[us]')

In [43]:
up_error = orderlines_df.loc[
    (orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+")) |
    (orderlines_df.unit_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]

In [44]:
up_error/orderlines_df.shape[0]*100

12.30309235568043

In [46]:
up_errorlist = orderlines_df.loc[
    (orderlines_df.unit_price.str.contains(r"\d+\.\d+\.\d+")) |
    (orderlines_df.unit_price.str.contains(r"\d+\.\d{3,}")), "id_order"]
orderlines_df = orderlines_df.loc[~orderlines_df.id_order.isin(up_errorlist)]

In [49]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"] )

In [50]:
orderlines_df.info()

<class 'pandas.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216250 non-null  int64         
 1   id_order          216250 non-null  int64         
 2   product_id        216250 non-null  int64         
 3   product_quantity  216250 non-null  int64         
 4   sku               216250 non-null  str           
 5   unit_price        216250 non-null  float64       
 6   date              216250 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4), str(1)
memory usage: 13.2 MB


In [51]:
orderlines_df.describe()

,id,id_order,product_id,product_quantity,unit_price,date
count,2.162500e+05,216250.000000,216250.0,216250.000000,216250.000000,216250
mean,1.372496e+06,408967.391727,0.0,1.136698,164.636160,2017-08-29 19:05:38.143486
min,1.119109e+06,241319.000000,0.0,1.000000,-119.000000,2017-01-01 00:07:19
25%,1.240903e+06,351866.250000,0.0,1.000000,29.740000,2017-05-03 17:36:03.250000
50%,1.366966e+06,406548.500000,0.0,1.000000,69.990000,2017-10-02 04:13:04.500000
75%,1.503947e+06,465795.500000,0.0,1.000000,184.580000,2017-12-18 23:35:52.250000
max,1.650203e+06,527401.000000,0.0,999.000000,999.990000,2018-03-14 13:58:36
std,1.517563e+05,65892.054168,0.0,3.292203,219.630604,NaN


# Products Table

8746 duplicate values - drop duplicates
desc - 7 missing, fill with name description
price - 46 missing and 596 decimal prob - to drop or Regex?
price - to numeric conversion
promo_price decimal value issue. also promo price is higher than price - to drop?
type - string and has scientific notation, no use

## Duplicates Check

In [52]:
products_df.duplicated().sum()

np.int64(8746)

In [54]:
products_df.loc[products_df.duplicated(keep=False),:].sort_values("sku", ascending = False)
# Checking the duplicate rows

,sku,name,desc,price,promo_price,in_stock,type
12723,WDT0317,WD My Cloud EX2 Ultra Mac and PC Server NAS,2-bay NAS 1GB RAM for Mac and PC,169.99,1.391.899,0,11935397
12724,WDT0317,WD My Cloud EX2 Ultra Mac and PC Server NAS,2-bay NAS 1GB RAM for Mac and PC,169.99,1.391.899,0,11935397
12716,WDT0311-A,Open - WD My Cloud EX2 Ultra Mac and PC Server...,2-bay NAS 1GB RAM for Mac and PC,169.99,1.325.138,1,11935397
12715,WDT0311-A,Open - WD My Cloud EX2 Ultra Mac and PC Server...,2-bay NAS 1GB RAM for Mac and PC,169.99,1.325.138,1,11935397
12714,WDT0311-A,Open - WD My Cloud EX2 Ultra Mac and PC Server...,2-bay NAS 1GB RAM for Mac and PC,169.99,1.325.138,1,11935397
...,...,...,...,...,...,...,...
107,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
108,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
101,APP0390,"Apple MacBook Pro 133 ""Core i5 25GHz | 4GB RAM...",MacBook Pro laptop 133 inches (MD101Y / A).,1199,11.455.917,0,1282
1902,AII0013,"Aiino Case MacBook Air 13 ""Transparent",MacBook Air 13-inch casing with matte finish.,39.99,179.927,0,13835403


In [55]:
products_df.drop_duplicates(inplace= True)

In [58]:
products_df["sku"].duplicated().sum()
# 'sku' is unique identifier so checking for duplicates

np.int64(1)

In [59]:
products_df.loc[products_df["sku"].duplicated(keep=False)]

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,NaN,1305.59,0,1282


In [60]:
products_df = products_df.drop(8000)
# deleting the duplicate value by drop(index)

## Missing Values

In [57]:
products_df.isnull().sum()

sku             0
name            0
desc            7
price          46
promo_price     0
in_stock        0
type           50
dtype: int64

In [61]:
products_df.loc[products_df['desc'].isna(), :]

,sku,name,desc,price,promo_price,in_stock,type
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN,107,814.659,0,1298
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN,1.568.206,1.568.206,0,1298
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN,566.35,5.659.896,0,12175397
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN,29.99,237.925,0,1298
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN,24.99,22.99,1,1515
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN,79.99,56.99,1,1515
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN,199.99,1.441.174,0,11905404


In [62]:
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']
# using the name column to fill the description

In [ ]:
products_df.loc[products_df['desc'].isna(), :]

In [65]:
products_df["price"].isnull().mean()*100

np.float64(0.4253710180546365)

In [67]:
products_df = products_df.dropna(subset=['price'])

In [68]:
products_df.info()

<class 'pandas.DataFrame'>
Index: 10534 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          10534 non-null  str  
 1   name         10534 non-null  str  
 2   desc         10534 non-null  str  
 3   price        10534 non-null  str  
 4   promo_price  10534 non-null  str  
 5   in_stock     10534 non-null  int64
 6   type         10484 non-null  str  
dtypes: int64(1), str(6)
memory usage: 658.4 KB


In [69]:
price_problems_number = products_df.loc[
    (products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
price_problems_number
# 5.15% of the rows of the DataFrame

542

Price is critical for accurate discount analysis, and this data will inform business decisions — so we're prioritizing trustworthiness over row retention. Deleting the malformed price rows.

In [70]:
products_df = products_df.loc[~((products_df.price.str.contains(r"\d+\.\d+\.\d+"))|
                                (products_df.price.str.contains(r"\d+\.\d{3,}"))), :]

## Datatype

In [71]:
products_df["price"] = pd.to_numeric(products_df["price"])

In [72]:
promo_problems_number = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
promo_problems_number

9232

In [73]:
round(((promo_problems_number / products_df.shape[0]) * 100), 2)
# promo_price has in total 9232 wrong values. This is 92.39% of the rows of the DataFrame
# This column appears to be very untrustworthy, we will delete the column.

92.39

In [74]:
products_cl = products_df.drop(columns=["promo_price"])

# Clean Dataset

In [76]:
orders_cl = orders_df.copy()
orderlines_cl = orderlines_df.copy()
products_cl = products_df.copy()
brands_cl = brands_df.copy()

In [79]:
orders_cl.to_csv("orders_cl.csv", index=False)
orderlines_cl.to_csv("orderlines_cl.csv", index=False)
products_cl.to_csv("products_cl.csv", index=False)
brands_cl.to_csv("brands_cl.csv", index=False)